# Evaluate every saved model

Run all cells with the **duckie-rl311** Python kernel and a running simulator.
This notebook calls `scripts/evaluate.py` once for every `models/*.zip` file,
including `baseline.zip`. Each call performs **five separate runs** by default.
The CSV contains one row per run, without averaging runs together.

Set the simulator port below (`7501 + simulator offset`). Use a simulator that
is free of training or other driving processes. Models run sequentially on the
same simulator using their saved speed and reward settings. The notebook does
not launch the simulator.

Results, raw JSON reports, and evaluator logs go into a new timestamped folder
under `training/logs/model_evaluations/`. See
[README_model_evaluation.md](README_model_evaluation.md) for column definitions.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import math
import subprocess
import sys
import zipfile

# Works when Jupyter starts from the repository root or scripts/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts/evaluate.py").is_file() and (path / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or scripts/.")


In [ ]:
# Configuration: run at least five evaluations per model.
RUNS_PER_MODEL = 5
DURATION_SECONDS = 60.0
PORT = 7501  # Confirmed running simulator; change if its port changes.
ENTITY = "map_0/vehicle_0"
DEVICE = "auto"  # auto, cpu, or cuda
SAME_START = False  # False keeps evaluate.py's five varied steering starts.
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = (
    PROJECT_ROOT / "training/logs/model_evaluations"
    / datetime.now().strftime("batch_%Y%m%d_%H%M%S_%f")
)

if not isinstance(RUNS_PER_MODEL, int) or RUNS_PER_MODEL < 5:
    raise ValueError("RUNS_PER_MODEL must be an integer of at least 5.")
if not math.isfinite(DURATION_SECONDS) or DURATION_SECONDS <= 0:
    raise ValueError("DURATION_SECONDS must be positive and finite.")
if not 1 <= PORT <= 65535:
    raise ValueError("PORT must be between 1 and 65535.")
if DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE must be auto, cpu, or cuda.")

model_paths = sorted(MODELS_DIR.glob("*.zip"))
if not model_paths:
    raise FileNotFoundError(f"No model ZIP files found in {MODELS_DIR}")
print(f"Models: {len(model_paths)}; expected CSV rows: {len(model_paths) * RUNS_PER_MODEL}")
print(f"Evaluation time: about {len(model_paths) * RUNS_PER_MODEL * DURATION_SECONDS / 3600:.2f} hours, plus loading/reset overhead")
print(f"Output: {OUTPUT_DIR}")


The first three CSV columns describe training: `learning_rate`, `speed`, and
`training_steps`. The next column, `steps`, is the number of steps measured in
that individual evaluation run. Action counts and termination reasons are
expanded into numeric columns; an absent count is zero.

The evaluator returns exit code 1 when a model fails its screening criteria.
Those runs still belong in the CSV. A process error or a missing run is reported
as an incomplete batch. Real results are saved after every model; missing runs
are never filled with invented measurements. Detailed progress is written to
each model's `.log` file while its five runs execute.

In [ ]:
ACTION_NAMES = ("right", "slight_right", "straight", "slight_left", "left")
METRIC_COLUMNS = (
    "steps", "reward_total", "reward_per_step", "both_lines_fraction",
    "longest_clean_streak",
)
DETAIL_COLUMNS = (
    "seconds", "requested_seconds", "fps", "failures", "time_limits",
    "mean_abs_center_error", "completed", "passed", "start_action", "offset_steps",
)
REASON_PREFIX = "termination_reason_count::"


def checkpoint_metadata(path):
    """Read the checkpoint's JSON metadata without loading weights or hardware."""
    with zipfile.ZipFile(path) as archive:
        data = json.loads(archive.read("data"))
    config = data["training_config"]
    # num_timesteps matches the saved checkpoint, including renamed baseline.zip.
    # In this project warm-start checkpoints count newly trained steps only.
    return {
        "learning_rate": float(config.get("LEARNING_RATE", data["learning_rate"])),
        "speed": float(config["MAX_SPEED"]),
        "training_steps": int(data["num_timesteps"]),
        "warm_start_steps": int(data.get("warm_start_steps", 0)),
    }


def report_rows(path, metadata, report):
    """Flatten each individual run, preserving the evaluator's metric values."""
    rows = []
    for number, run in enumerate(report.get("runs", []), start=1):
        row = {key: metadata[key] for key in ("learning_rate", "speed", "training_steps")}
        row.update({key: run[key] for key in METRIC_COLUMNS})
        row.update({f"action_count_{name}": run["actions"].get(name, 0)
                    for name in ACTION_NAMES})
        reasons = run["termination_reasons"]
        row["termination_reason_count"] = sum(reasons.values())
        row.update({REASON_PREFIX + reason: count for reason, count in reasons.items()})
        row.update(model=path.name, run=number, warm_start_steps=metadata["warm_start_steps"])
        row.update({key: run.get(key) for key in DETAIL_COLUMNS})
        rows.append(row)
    return rows


def write_csv(rows, path):
    """Include all observed termination reasons and zero-fill missing counts."""
    reason_columns = sorted({key for row in rows for key in row if key.startswith(REASON_PREFIX)})
    columns = [
        "learning_rate", "speed", "training_steps", *METRIC_COLUMNS,
        *(f"action_count_{name}" for name in ACTION_NAMES),
        "termination_reason_count", *reason_columns,
        "model", "run", "warm_start_steps", *DETAIL_COLUMNS,
    ]
    # Replace atomically so a reader never sees a half-written CSV.
    temporary = path.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, 0 if key in reason_columns else "")
                             for key in columns})
    temporary.replace(path)


def evaluate_models(model_paths, output_dir, *, runs, duration, port, entity, device, same_start):
    """Call evaluate.py sequentially and checkpoint the CSV after each model."""
    if not isinstance(runs, int) or runs < 5:
        raise ValueError("At least five runs per model are required.")
    output_dir.mkdir(parents=True, exist_ok=True)
    reports_dir = output_dir / "reports"
    reports_dir.mkdir(exist_ok=True)
    csv_path = output_dir / "evaluations.csv"
    rows, issues = [], []
    write_csv(rows, csv_path)
    for index, path in enumerate(model_paths, start=1):
        report_path = reports_dir / f"{path.stem}.json"
        log_path = reports_dir / f"{path.stem}.log"
        # Re-running this cell evaluates again; never consume an old report.
        report_path.unlink(missing_ok=True)
        print(f"[{index}/{len(model_paths)}] {path.name}", flush=True)
        print(f"  Live log: {log_path}", flush=True)
        try:
            metadata = checkpoint_metadata(path)
            command = [
                sys.executable, "-u", str(PROJECT_ROOT / "scripts/evaluate.py"),
                "--model", str(path.resolve()), "--runs", str(runs),
                "--duration", str(duration), "--port", str(port), "--entity", entity,
                "--device", device, "--output", str(report_path.resolve()),
            ]
            if same_start:
                command.append("--same-start")
            with log_path.open("w", encoding="utf-8") as log:
                process = subprocess.run(command, cwd=PROJECT_ROOT, stdout=log,
                                         stderr=subprocess.STDOUT, check=False)
            if not report_path.is_file():
                raise RuntimeError(f"evaluate.py exited {process.returncode} without a JSON report")
            report = json.loads(report_path.read_text(encoding="utf-8"))
            model_rows = report_rows(path, metadata, report)
            rows.extend(model_rows)
            problems = []
            if process.returncode not in (0, 1):
                problems.append(f"exit code {process.returncode}")
            if report.get("error"):
                problems.append(report["error"])
            if len(model_rows) != runs:
                problems.append(f"expected {runs} runs, received {len(model_rows)}")
            if any(not row["completed"] for row in model_rows):
                problems.append("one or more evaluation runs did not complete")
            if problems:
                raise RuntimeError("; ".join(problems))
            print(f"  Saved {len(model_rows)} runs; screening passed: {report.get('passed')}", flush=True)
        except Exception as error:
            issue = {"model": path.name, "error": f"{type(error).__name__}: {error}",
                     "log": str(log_path)}
            issues.append(issue)
            print(f"  INCOMPLETE: {issue['error']}", flush=True)
        finally:
            write_csv(rows, csv_path)
            (output_dir / "issues.json").write_text(json.dumps(issues, indent=2) + "\n", encoding="utf-8")
    return rows, issues


In [ ]:
# This cell drives the simulator. It can take several hours for the whole folder.
rows, issues = evaluate_models(
    model_paths, OUTPUT_DIR, runs=RUNS_PER_MODEL, duration=DURATION_SECONDS,
    port=PORT, entity=ENTITY, device=DEVICE, same_start=SAME_START,
)

csv_path = OUTPUT_DIR / "evaluations.csv"
print(f"Saved {len(rows)} individual evaluation rows to {csv_path}")
if issues:
    raise RuntimeError(
        f"Batch incomplete for {len(issues)} model(s). Real results are saved in {csv_path}; "
        f"see {OUTPUT_DIR / 'issues.json'} and the model logs."
    )
assert len(rows) == len(model_paths) * RUNS_PER_MODEL
assert all(sum(row["model"] == path.name for row in rows) == RUNS_PER_MODEL
           for path in model_paths)
print(f"Complete: every model has {RUNS_PER_MODEL} CSV rows.")


In [ ]:
# Preview the saved file (also works after an incomplete batch).
import pandas as pd

results = pd.read_csv(OUTPUT_DIR / "evaluations.csv")
display(results)
display(results.groupby("model", sort=False).size().rename("evaluation_runs").to_frame())
